# Phase 1 — Weather aggregation (10 km grid)

**Phase 1 screening:** Rich distribution/shape features (~80–100 columns).

| Input | Output |
|-------|--------|
| `data/raw_solar_wind_hourly_data/NY_State_10km_YYYY.csv` | `data/processed/weather_cell_features.parquet` |

**Wind:** Hub-height speed (80 m, α=0.14), hub power proxy, percentiles p50–p95, ramp p95, Weibull k/A, DJF/JJA hub percentiles. Legacy 10 m columns retained.

**Solar:** Daytime means (unchanged), diurnal amplitude, clear-sky index, direct/diffuse ratio, monthly shortwave std.

**Data quality:** Bad rows logged to `weather_quarantine_log.csv` (no silent drops).

> After changing feature logic, set `INVALIDATE_SCRATCH = True` and re-run `01` → `05` → `06` → `07`–`15`.

> **Tip:** For a quick test, set `YEARS = [2025]` first (~2 GB per year). Then set `YEARS = None` for all years.

In [1]:
# --- User settings ---
YEARS = None  # None = all years in config
USE_SCRATCH_CACHE = True  # reuse data/processed/scratch/weather_cell_year/*.parquet
INVALIDATE_SCRATCH = False  # set True only when feature logic changes

import shutil
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import polars as pl
import yaml

ROOT = Path('../..').resolve()
CONFIG_PATH = ROOT / 'config' / 'paths.yaml'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'weather_cell_features.parquet'
SCRATCH_DIR = ROOT / 'data' / 'processed' / 'scratch' / 'weather_cell_year'
QUARANTINE_PATH = ROOT / 'data' / 'processed' / 'weather_quarantine_log.csv'

with CONFIG_PATH.open(encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

GRID_DIR = ROOT / CONFIG['data']['raw']['grid_weather_dir']
YEARS = YEARS or CONFIG['weather']['grid_years']
WIND_SOURCE = CONFIG['weather']['wind_source']

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if INVALIDATE_SCRATCH and SCRATCH_DIR.exists():
    shutil.rmtree(SCRATCH_DIR)
    SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
    print('Invalidated scratch cache:', SCRATCH_DIR)

print('Repo root :', ROOT)
print('Grid dir  :', GRID_DIR)
print('Years       :', YEARS)
print('Wind source :', WIND_SOURCE)

Invalidated scratch cache: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/scratch/weather_cell_year
Repo root : /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York
Grid dir  : /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/raw_solar_wind_hourly_data
Years       : [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Wind source : grid_10km


In [2]:
import math

import numpy as np
from scipy.optimize import brentq
from scipy.special import gamma as gammafn

SEASONS = ('DJF', 'MAM', 'JJA', 'SON')
MONTHLY_SHORTWAVE_COLS = [f'shortwave_mean_m{m:02d}' for m in range(1, 13)]
MONTHLY_WIND_COLS = [f'wind_speed_mean_m{m:02d}' for m in range(1, 13)]
MONTHLY_SHORTWAVE_STD_COLS = [f'shortwave_std_m{m:02d}' for m in range(1, 13)]
SEASONAL_FEATURE_COLS = [
    f'{prefix}_{season}'
    for prefix in ('shortwave_mean', 'wind_speed_mean', 'wind_power_mean')
    for season in SEASONS
]
SEASONAL_HUB_PERCENTILE_COLS = [
    f'wind_hub_speed_{pct}_{season}'
    for pct in ('p50', 'p90')
    for season in ('DJF', 'JJA')
]

HUB_HEIGHT_M = 80.0
WIND_SHEAR_ALPHA = 0.14
HUB_SPEED_FACTOR = (HUB_HEIGHT_M / 10.0) ** WIND_SHEAR_ALPHA
DAYLIGHT_SHORTWAVE_THRESHOLD = 0.0

YEAR_FRAME_COLS = [
    'latitude', 'longitude',
    'shortwave_mean', 'shortwave_mean_24h', 'shortwave_std', 'direct_mean', 'diffuse_mean',
    'shortwave_diurnal_amplitude', 'clear_sky_index', 'direct_diffuse_ratio',
    'wind_speed_mean', 'wind_speed_std', 'wind_speed_p90', 'wind_power_mean',
    'wind_speed_hub_mean', 'wind_speed_hub_std', 'wind_hub_power_mean',
    'wind_hub_speed_p50', 'wind_hub_speed_p75', 'wind_hub_speed_p90', 'wind_hub_speed_p95',
    'wind_ramp_p95', 'weibull_k', 'weibull_A',
    'temperature_mean', 'surface_pressure_mean', 'year',
    *MONTHLY_SHORTWAVE_COLS,
    *MONTHLY_SHORTWAVE_STD_COLS,
    *MONTHLY_WIND_COLS,
    *SEASONAL_FEATURE_COLS,
    *SEASONAL_HUB_PERCENTILE_COLS,
]

SEASON_EXPR = (
    pl.when(pl.col('month').is_in([12, 1, 2])).then(pl.lit('DJF'))
    .when(pl.col('month').is_in([3, 4, 5])).then(pl.lit('MAM'))
    .when(pl.col('month').is_in([6, 7, 8])).then(pl.lit('JJA'))
    .otherwise(pl.lit('SON'))
    .alias('season')
)


def weibull_k_from_cv(cv: float) -> float:
    def cv_for_k(k: float) -> float:
        return math.sqrt(gammafn(1 + 2 / k) / gammafn(1 + 1 / k) ** 2 - 1)

    return brentq(lambda k: cv_for_k(k) - cv, 0.05, 20.0)


def weibull_params(mean: float, std: float) -> tuple[float, float]:
    if not np.isfinite(mean) or not np.isfinite(std) or mean <= 0 or std <= 0:
        return float('nan'), float('nan')
    cv = std / mean
    if cv <= 0:
        return float('nan'), float('nan')
    try:
        k = weibull_k_from_cv(cv)
        scale = mean / gammafn(1 + 1 / k)
        return float(k), float(scale)
    except (ValueError, ZeroDivisionError):
        return float('nan'), float('nan')


def clear_sky_ghi_w_m2(latitude: pl.Expr, doy: pl.Expr) -> pl.Expr:
    lat_rad = latitude * (math.pi / 180.0)
    decl = pl.lit(23.45 * math.pi / 180.0) * ((doy - 81) * pl.lit(2 * math.pi / 365.0)).sin()
    sin_elev_max = lat_rad.sin() * decl.sin() + lat_rad.cos() * decl.cos()
    return (pl.lit(1361.0) * pl.lit(0.65) * sin_elev_max).clip(lower_bound=1.0)


def load_hourly_with_quarantine(
    csv_path: Path,
    year: int,
    quarantine_frames: list[pl.DataFrame],
) -> pl.LazyFrame:
    raw = pl.scan_csv(
        csv_path,
        try_parse_dates=False,
        ignore_errors=True,
        infer_schema_length=10_000,
    ).collect()

    parsed = raw.with_columns(
        pl.col('date')
        .str.strip_chars()
        .str.to_datetime(time_zone='UTC', strict=False)
        .alias('date_parsed'),
        pl.lit(year).alias('source_year'),
        pl.lit(csv_path.name).alias('source_file'),
    )

    bad = parsed.filter(
        pl.col('date_parsed').is_null() | pl.col('latitude').is_null() | pl.col('longitude').is_null()
    )
    if bad.height:
        quarantine_frames.append(
            bad.select(['source_year', 'source_file', 'date', 'latitude', 'longitude'])
        )

    return (
        parsed.filter(
            pl.col('date_parsed').is_not_null()
            & pl.col('latitude').is_not_null()
            & pl.col('longitude').is_not_null()
        )
        .with_columns(pl.col('date_parsed').alias('date'))
        .drop('date_parsed')
        .lazy()
    )


def align_year_frame(df: pl.DataFrame) -> pl.DataFrame:
    missing = [c for c in YEAR_FRAME_COLS if c not in df.columns]
    if missing:
        df = df.with_columns([pl.lit(None).cast(pl.Float64).alias(c) for c in missing])
    return df.select(YEAR_FRAME_COLS)


def enrich_hourly(lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        lf.with_columns(
            pl.col('shortwave_radiation').clip(lower_bound=0).alias('shortwave_radiation'),
            pl.col('direct_radiation').clip(lower_bound=0).alias('direct_radiation'),
            pl.col('diffuse_radiation').clip(lower_bound=0).alias('diffuse_radiation'),
        )
        .with_columns(
            pl.when(pl.col('shortwave_radiation') > DAYLIGHT_SHORTWAVE_THRESHOLD)
            .then(pl.col('shortwave_radiation'))
            .otherwise(None)
            .alias('shortwave_radiation_daytime'),
            pl.when(
                (pl.col('diffuse_radiation') > 0)
                & (pl.col('shortwave_radiation') > DAYLIGHT_SHORTWAVE_THRESHOLD)
            )
            .then(pl.col('direct_radiation') / pl.col('diffuse_radiation'))
            .otherwise(None)
            .alias('direct_diffuse_ratio_hourly'),
        )
        .with_columns(
            pl.col('date').dt.month().alias('month'),
            pl.col('date').dt.ordinal_day().alias('doy'),
        )
        .with_columns(
            (pl.col('wind_speed_10m') * HUB_SPEED_FACTOR).alias('wind_speed_hub'),
            (pl.col('wind_speed_10m').pow(3)).alias('wind_power_proxy'),
            (pl.col('wind_speed_10m') * HUB_SPEED_FACTOR).pow(3).alias('wind_hub_power_proxy'),
            clear_sky_ghi_w_m2(pl.col('latitude'), pl.col('doy')).alias('clear_sky_ghi'),
            SEASON_EXPR,
        )
        .with_columns(
            pl.when(pl.col('shortwave_radiation_daytime').is_not_null())
            .then(pl.col('shortwave_radiation_daytime') / pl.col('clear_sky_ghi'))
            .otherwise(None)
            .alias('clear_sky_index_hourly'),
        )
        .sort(['latitude', 'longitude', 'date'])
        .with_columns(
            pl.col('wind_speed_hub').diff().over(['latitude', 'longitude']).abs().alias('wind_ramp'),
        )
    )


def monthly_means(lf: pl.LazyFrame, value_col: str, prefix: str) -> pl.DataFrame:
    monthly = (
        lf.group_by(['latitude', 'longitude', 'month'])
        .agg(pl.col(value_col).mean().alias('value'))
        .collect()
    )
    wide = monthly.pivot(on='month', index=['latitude', 'longitude'], values='value')
    rename_map = {str(m): f'{prefix}_m{m:02d}' for m in range(1, 13) if str(m) in wide.columns}
    return wide.rename(rename_map)


def monthly_stds(lf: pl.LazyFrame, value_col: str, prefix: str) -> pl.DataFrame:
    monthly = (
        lf.group_by(['latitude', 'longitude', 'month'])
        .agg(pl.col(value_col).std().alias('value'))
        .collect()
    )
    wide = monthly.pivot(on='month', index=['latitude', 'longitude'], values='value')
    rename_map = {str(m): f'{prefix}_m{m:02d}' for m in range(1, 13) if str(m) in wide.columns}
    return wide.rename(rename_map)


def seasonal_means(lf: pl.LazyFrame, value_col: str, prefix: str) -> pl.DataFrame:
    seasonal = (
        lf.group_by(['latitude', 'longitude', 'season'])
        .agg(pl.col(value_col).mean().alias('value'))
        .collect()
    )
    wide = seasonal.pivot(on='season', index=['latitude', 'longitude'], values='value')
    rename_map = {s: f'{prefix}_{s}' for s in SEASONS if s in wide.columns}
    return wide.rename(rename_map)


def seasonal_hub_percentiles(lf: pl.LazyFrame, pct: float, suffix: str) -> pl.DataFrame:
    seasonal = (
        lf.group_by(['latitude', 'longitude', 'season'])
        .agg(pl.col('wind_speed_hub').quantile(pct).alias('value'))
        .collect()
    )
    wide = seasonal.pivot(on='season', index=['latitude', 'longitude'], values='value')
    rename_map = {
        s: f'wind_hub_speed_{suffix}_{s}' for s in ('DJF', 'JJA') if s in wide.columns
    }
    return wide.rename(rename_map)


def aggregate_year(
    csv_path: Path,
    year: int,
    quarantine_frames: list[pl.DataFrame],
) -> pl.DataFrame:
    hourly = load_hourly_with_quarantine(csv_path, year, quarantine_frames)
    lf = enrich_hourly(hourly)

    annual = lf.group_by(['latitude', 'longitude']).agg(
        pl.col('shortwave_radiation_daytime').mean().alias('shortwave_mean'),
        pl.col('shortwave_radiation').mean().alias('shortwave_mean_24h'),
        pl.col('shortwave_radiation').std().alias('shortwave_std'),
        pl.col('direct_radiation').mean().alias('direct_mean'),
        pl.col('diffuse_radiation').mean().alias('diffuse_mean'),
        (
            pl.col('shortwave_radiation_daytime').max() - pl.col('shortwave_radiation_daytime').min()
        ).alias('shortwave_diurnal_amplitude'),
        pl.col('clear_sky_index_hourly').mean().alias('clear_sky_index'),
        pl.col('direct_diffuse_ratio_hourly').mean().alias('direct_diffuse_ratio'),
        pl.col('wind_speed_10m').mean().alias('wind_speed_mean'),
        pl.col('wind_speed_10m').std().alias('wind_speed_std'),
        pl.col('wind_speed_10m').quantile(0.9).alias('wind_speed_p90'),
        pl.col('wind_power_proxy').mean().alias('wind_power_mean'),
        pl.col('wind_speed_hub').mean().alias('wind_speed_hub_mean'),
        pl.col('wind_speed_hub').std().alias('wind_speed_hub_std'),
        pl.col('wind_hub_power_proxy').mean().alias('wind_hub_power_mean'),
        pl.col('wind_speed_hub').quantile(0.5).alias('wind_hub_speed_p50'),
        pl.col('wind_speed_hub').quantile(0.75).alias('wind_hub_speed_p75'),
        pl.col('wind_speed_hub').quantile(0.9).alias('wind_hub_speed_p90'),
        pl.col('wind_speed_hub').quantile(0.95).alias('wind_hub_speed_p95'),
        pl.col('wind_ramp').quantile(0.95).alias('wind_ramp_p95'),
        pl.col('temperature_2m').mean().alias('temperature_mean'),
        pl.col('surface_pressure').mean().alias('surface_pressure_mean'),
        pl.lit(year).alias('year'),
    ).collect()

    hub_means = annual['wind_speed_hub_mean'].to_numpy()
    hub_stds = annual['wind_speed_hub_std'].to_numpy()
    weibull_k = np.empty(len(annual), dtype=np.float64)
    weibull_a = np.empty(len(annual), dtype=np.float64)
    for i, (mean, std) in enumerate(zip(hub_means, hub_stds, strict=True)):
        weibull_k[i], weibull_a[i] = weibull_params(float(mean), float(std))
    annual = annual.with_columns(
        pl.Series('weibull_k', weibull_k),
        pl.Series('weibull_A', weibull_a),
    )

    parts = [
        monthly_means(lf, 'shortwave_radiation_daytime', 'shortwave_mean'),
        monthly_stds(lf, 'shortwave_radiation_daytime', 'shortwave_std'),
        monthly_means(lf, 'wind_speed_10m', 'wind_speed_mean'),
        seasonal_means(lf, 'shortwave_radiation_daytime', 'shortwave_mean'),
        seasonal_means(lf, 'wind_speed_10m', 'wind_speed_mean'),
        seasonal_means(lf, 'wind_power_proxy', 'wind_power_mean'),
        seasonal_hub_percentiles(lf, 0.5, 'p50'),
        seasonal_hub_percentiles(lf, 0.9, 'p90'),
    ]
    result = annual
    for part in parts:
        result = result.join(part, on=['latitude', 'longitude'], how='left')
    return align_year_frame(result)


def combine_years(year_frames: list[pl.DataFrame]) -> pl.DataFrame:
    aligned = [align_year_frame(df) for df in year_frames]
    combined = pl.concat(aligned, how='vertical')
    agg_exprs = [
        pl.col('shortwave_mean').mean().alias('shortwave_mean'),
        pl.col('shortwave_mean_24h').mean().alias('shortwave_mean_24h'),
        pl.col('shortwave_mean').std().alias('shortwave_interannual_std'),
        pl.col('shortwave_std').mean().alias('shortwave_intraseasonal_std_mean'),
        pl.col('direct_mean').mean().alias('direct_mean'),
        pl.col('diffuse_mean').mean().alias('diffuse_mean'),
        pl.col('shortwave_diurnal_amplitude').mean().alias('shortwave_diurnal_amplitude'),
        pl.col('clear_sky_index').mean().alias('clear_sky_index'),
        pl.col('direct_diffuse_ratio').mean().alias('direct_diffuse_ratio'),
        pl.col('wind_speed_mean').mean().alias('wind_speed_mean'),
        pl.col('wind_speed_mean').std().alias('wind_speed_interannual_std'),
        pl.col('wind_speed_std').mean().alias('wind_speed_intraseasonal_std_mean'),
        pl.col('wind_speed_p90').mean().alias('wind_speed_p90'),
        pl.col('wind_power_mean').mean().alias('wind_power_mean'),
        pl.col('wind_speed_hub_mean').mean().alias('wind_speed_hub_mean'),
        pl.col('wind_speed_hub_std').mean().alias('wind_speed_hub_std'),
        pl.col('wind_hub_power_mean').mean().alias('wind_hub_power_mean'),
        pl.col('wind_hub_speed_p50').mean().alias('wind_hub_speed_p50'),
        pl.col('wind_hub_speed_p75').mean().alias('wind_hub_speed_p75'),
        pl.col('wind_hub_speed_p90').mean().alias('wind_hub_speed_p90'),
        pl.col('wind_hub_speed_p95').mean().alias('wind_hub_speed_p95'),
        pl.col('wind_ramp_p95').mean().alias('wind_ramp_p95'),
        pl.col('weibull_k').mean().alias('weibull_k'),
        pl.col('weibull_A').mean().alias('weibull_A'),
        pl.col('temperature_mean').mean().alias('temperature_mean'),
        pl.col('surface_pressure_mean').mean().alias('surface_pressure_mean'),
        pl.col('year').n_unique().alias('n_years'),
        pl.col('year').min().alias('year_start'),
        pl.col('year').max().alias('year_end'),
    ]
    extra_cols = (
        MONTHLY_SHORTWAVE_COLS
        + MONTHLY_SHORTWAVE_STD_COLS
        + MONTHLY_WIND_COLS
        + SEASONAL_FEATURE_COLS
        + SEASONAL_HUB_PERCENTILE_COLS
    )
    for col in extra_cols:
        agg_exprs.append(pl.col(col).mean().alias(col))
    return combined.group_by(['latitude', 'longitude']).agg(agg_exprs)


def write_quarantine_log(quarantine_frames: list[pl.DataFrame], path: Path) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not quarantine_frames:
        pl.DataFrame(
            schema={
                'source_year': pl.Int32,
                'source_file': pl.Utf8,
                'date': pl.Utf8,
                'latitude': pl.Float64,
                'longitude': pl.Float64,
            }
        ).write_csv(path)
        return 0
    log = pl.concat(quarantine_frames, how='vertical_relaxed')
    log.write_csv(path)
    return log.height


print(f'Aggregation helpers ready — {len(YEAR_FRAME_COLS)} year-level columns')

Aggregation helpers ready — 79 year-level columns


In [3]:
year_frames: list[pl.DataFrame] = []
quarantine_frames: list[pl.DataFrame] = []
cells_per_year: dict[int, int] = {}

for year in YEARS:
    csv_path = GRID_DIR / f'NY_State_10km_{year}.csv'
    if not csv_path.exists():
        raise FileNotFoundError(f'Missing grid weather file: {csv_path}')

    scratch_file = SCRATCH_DIR / f'weather_cell_{year}.parquet'
    if USE_SCRATCH_CACHE and scratch_file.exists():
        print(f'Loading cached {scratch_file.name}')
        year_df = align_year_frame(pl.read_parquet(scratch_file))
    else:
        print(f'Aggregating {csv_path.name} ...')
        year_df = aggregate_year(csv_path, year, quarantine_frames)
        year_df.write_parquet(scratch_file)
        print(f'  -> {len(year_df):,} cells cached to {scratch_file.name}')

    cells_per_year[year] = len(year_df)
    year_frames.append(year_df)

n_quarantined = write_quarantine_log(quarantine_frames, QUARANTINE_PATH)
features = combine_years(year_frames)
features = features.with_columns(pl.lit(WIND_SOURCE).alias('wind_data_source'))
features.write_parquet(OUTPUT_PATH)

print(f'\nWrote {len(features):,} cells x {len(features.columns)} columns')
print('Output:', OUTPUT_PATH)
print('Quarantine log:', QUARANTINE_PATH, f'({n_quarantined:,} bad rows)')
print('Cells per year:', cells_per_year)
if any(c < 4000 for c in cells_per_year.values()):
    print('NOTE: 2024/2025 have fewer cells than 2018-2023 — union still yields full grid in combine_years')

Aggregating NY_State_10km_2018.csv ...


  -> 3,034 cells cached to weather_cell_2018.parquet
Aggregating NY_State_10km_2019.csv ...


  -> 3,030 cells cached to weather_cell_2019.parquet
Aggregating NY_State_10km_2020.csv ...


  -> 3,033 cells cached to weather_cell_2020.parquet
Aggregating NY_State_10km_2021.csv ...


  -> 4,399 cells cached to weather_cell_2021.parquet
Aggregating NY_State_10km_2022.csv ...


  -> 3,040 cells cached to weather_cell_2022.parquet
Aggregating NY_State_10km_2023.csv ...


  -> 4,400 cells cached to weather_cell_2023.parquet
Aggregating NY_State_10km_2024.csv ...


  -> 3,030 cells cached to weather_cell_2024.parquet
Aggregating NY_State_10km_2025.csv ...


  -> 3,033 cells cached to weather_cell_2025.parquet

Wrote 4,401 cells x 84 columns
Output: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/weather_cell_features.parquet
Quarantine log: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/weather_quarantine_log.csv (2 bad rows)
Cells per year: {2018: 3034, 2019: 3030, 2020: 3033, 2021: 4399, 2022: 3040, 2023: 4400, 2024: 3030, 2025: 3033}
NOTE: 2024/2025 have fewer cells than 2018-2023 — union still yields full grid in combine_years


In [4]:
# Quick preview
preview = pl.read_parquet(OUTPUT_PATH)
print(preview.schema)
preview.head(5)

Schema({'latitude': Float64, 'longitude': Float64, 'shortwave_mean': Float64, 'shortwave_mean_24h': Float64, 'shortwave_interannual_std': Float64, 'shortwave_intraseasonal_std_mean': Float64, 'direct_mean': Float64, 'diffuse_mean': Float64, 'shortwave_diurnal_amplitude': Float64, 'clear_sky_index': Float64, 'direct_diffuse_ratio': Float64, 'wind_speed_mean': Float64, 'wind_speed_interannual_std': Float64, 'wind_speed_intraseasonal_std_mean': Float64, 'wind_speed_p90': Float64, 'wind_power_mean': Float64, 'wind_speed_hub_mean': Float64, 'wind_speed_hub_std': Float64, 'wind_hub_power_mean': Float64, 'wind_hub_speed_p50': Float64, 'wind_hub_speed_p75': Float64, 'wind_hub_speed_p90': Float64, 'wind_hub_speed_p95': Float64, 'wind_ramp_p95': Float64, 'weibull_k': Float64, 'weibull_A': Float64, 'temperature_mean': Float64, 'surface_pressure_mean': Float64, 'n_years': UInt32, 'year_start': Int32, 'year_end': Int32, 'shortwave_mean_m01': Float64, 'shortwave_mean_m02': Float64, 'shortwave_mean_m

latitude,longitude,shortwave_mean,shortwave_mean_24h,shortwave_interannual_std,shortwave_intraseasonal_std_mean,direct_mean,diffuse_mean,shortwave_diurnal_amplitude,clear_sky_index,direct_diffuse_ratio,wind_speed_mean,wind_speed_interannual_std,wind_speed_intraseasonal_std_mean,wind_speed_p90,wind_power_mean,wind_speed_hub_mean,wind_speed_hub_std,wind_hub_power_mean,wind_hub_speed_p50,wind_hub_speed_p75,wind_hub_speed_p90,wind_hub_speed_p95,wind_ramp_p95,weibull_k,weibull_A,temperature_mean,surface_pressure_mean,n_years,year_start,year_end,shortwave_mean_m01,shortwave_mean_m02,shortwave_mean_m03,shortwave_mean_m04,shortwave_mean_m05,shortwave_mean_m06,…,shortwave_std_m05,shortwave_std_m06,shortwave_std_m07,shortwave_std_m08,shortwave_std_m09,shortwave_std_m10,shortwave_std_m11,shortwave_std_m12,wind_speed_mean_m01,wind_speed_mean_m02,wind_speed_mean_m03,wind_speed_mean_m04,wind_speed_mean_m05,wind_speed_mean_m06,wind_speed_mean_m07,wind_speed_mean_m08,wind_speed_mean_m09,wind_speed_mean_m10,wind_speed_mean_m11,wind_speed_mean_m12,shortwave_mean_DJF,shortwave_mean_MAM,shortwave_mean_JJA,shortwave_mean_SON,wind_speed_mean_DJF,wind_speed_mean_MAM,wind_speed_mean_JJA,wind_speed_mean_SON,wind_power_mean_DJF,wind_power_mean_MAM,wind_power_mean_JJA,wind_power_mean_SON,wind_hub_speed_p50_DJF,wind_hub_speed_p50_JJA,wind_hub_speed_p90_DJF,wind_hub_speed_p90_JJA,wind_data_source
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,i32,i32,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
41.85,-72.42,308.084773,164.453651,10.726929,241.597051,106.127083,58.326568,968.375,0.468552,1.881404,3.350913,0.147335,1.551727,5.4519808,65.189832,4.483279,2.076099,156.12687,4.174565,5.673169,7.294355,8.379069,1.564502,2.293051,5.060428,10.211249,992.533432,8,2018,2025,194.068145,243.078278,320.376564,334.67874,375.286649,380.407452,…,289.788933,289.69759,281.577252,260.862176,246.263326,201.396025,167.050373,140.599125,3.763249,3.630374,3.92755,3.819377,3.273389,2.957085,2.719491,2.703256,2.842762,3.30643,3.666949,3.623664,201.39643,345.331964,376.051941,267.780072,3.673426,3.671852,2.791497,3.272425,84.666089,82.931874,34.577023,58.899474,4.625914,3.606523,7.973518,5.832621,"""grid_10km"""
41.22,-71.97,326.402946,174.254233,9.582264,252.731561,116.108793,58.145441,969.125,0.490363,2.040113,6.645459,0.250052,3.139405,10.863449,512.748771,8.891143,4.200296,1228.011469,8.391084,11.457872,14.534508,16.557503,2.556959,2.240111,10.03825,11.425797,1016.571981,8,2018,2025,201.279435,245.074259,335.817557,355.853728,396.525171,407.974734,…,296.663438,302.154532,292.801516,274.569503,254.628785,210.462784,171.369837,142.437046,7.946263,7.293887,7.446706,6.940918,6.012358,5.381421,4.970123,5.218237,5.824248,7.014563,7.843843,7.895069,206.860936,364.7481,403.845986,284.382641,7.723579,6.798463,5.187846,6.89554,759.639367,515.771091,225.564624,555.189346,9.978303,6.666913,16.467406,11.003484,"""grid_10km"""
41.22,-72.15,322.539828,172.19187,10.229151,249.714065,113.603537,58.588332,968.25,0.485369,1.97887,3.976458,0.15561,1.875507,6.434191,110.909812,5.320213,2.509293,265.624276,4.997109,6.746204,8.608482,9.864757,1.75081,2.244161,6.006545,11.647399,1016.614547,8,2018,2025,201.916532,247.077668,332.30633,350.20982,391.120324,401.983566,…,293.557655,298.563736,289.664301,269.079955,252.464702,209.036663,170.771436,142.789968,4.49637,4.269427,4.528197,4.361541,3.771882,3.42661,3.21591,3.28564,3.496394,4.080886,4.387635,4.417689,207.640094,359.834154,396.857418,281.691414,4.397776,4.219007,3.308112,3.989323,150.910396,125.240839,58.746103,109.489925,5.546874,4.266237,9.556712,6.96449,"""grid_10km"""
43.11,-73.68,297.80517,159.223933,10.510917,235.630136,100.302008,58.921925,959.75,0.456511,1.711338,2.606804,0.169856,1.325004,4.370878,33.400112,3.487715,1.77276,79.991846,3.326778,4

In [5]:
# Seasonal sanity check — NY should show higher summer solar and higher winter wind on average
season_cols = [c for c in preview.columns if c.endswith(('DJF', 'MAM', 'JJA', 'SON'))]
preview.select(
    [c for c in season_cols if c.startswith(('shortwave_mean', 'wind_speed_mean'))]
).mean()

shortwave_mean_DJF,shortwave_mean_MAM,shortwave_mean_JJA,shortwave_mean_SON,wind_speed_mean_DJF,wind_speed_mean_MAM,wind_speed_mean_JJA,wind_speed_mean_SON
f64,f64,f64,f64,f64,f64,f64,f64
174.166185,340.748384,381.063002,255.547459,3.809767,3.695691,2.865478,3.400463
